In [2]:
import json
import pandas as pd
import xarray as xr
import numpy as np
import geopandas as gpd

In [3]:
records = []
list_years = list(range(2012, 2025))
for year in list_years:
    with open(f"./resources/GFW/GFW_unprocessed/GFW_SWA_AIS_{year}.json") as f:
        raw = json.load(f)

    # Extract the list of vessel records
    entries = raw["entries"][0]["public-global-fishing-effort:v3.0"]
    records.extend(entries)  # Append to the main records list

    df1 = pd.DataFrame(records)

In [4]:
df1.head()

,callsign,dataset,date,entryTimestamp,exitTimestamp,firstTransmissionDate,flag,geartype,hours,imo,lastTransmissionDate,lat,lon,mmsi,shipName,vesselId,vesselType
0,6672,public-global-vessel-identity:v3.0,2012-03,2012-02-19T14:00:00Z,2012-12-16T14:00:00Z,2012-01-16T21:30:32Z,ARG,TRAWLERS,1.633333,,2016-10-15T19:42:34Z,-40.8,-59.099998,701000543,KARINA,9862acd0c-caec-5efc-3cb4-28fb8d02a840,FISHING
1,ZDLB2,public-global-vessel-identity:v3.0,2012-02,2012-02-06T14:00:00Z,2012-11-08T14:00:00Z,2012-01-13T08:07:05Z,FLK,TRAWLERS,4.257778,8517463,2018-10-14T19:24:02Z,-52.4,-58.099998,740368000,KALATXORI,222dbf89f-faa5-4a3a-247d-0fd25383cd28,FISHING
2,LW8399,public-global-vessel-identity:v3.0,2012-07,2012-03-18T15:00:00Z,2012-11-27T14:00:00Z,2012-01-01T06:10:17Z,ARG,TRAWLERS,6.658056,,2018-05-07T19:53:38Z,-45.2,-63.000000,701000660,FLORIDA BLANCA IV,962d6b22a-ac9a-60ce-3129-3e06470f2a67,FISHING
3,LW 2962,public-global-vessel-identity:v3.0,2012-10,2012-04-25T04:00:00Z,2012-11-27T08:00:00Z,2012-03-08T01:16:30Z,ARG,TRAWLERS,11.311944,,2013-01-18T18:16:10Z,-44.6,-64.400002,701000619,SUEMAR DOS,6d762bbb1-1ae7-bc31-e5fb-568b8e462c3c,FISHING
4,LW2618,public-global-vessel-identity:v3.0,2012-03,2012-03-24T02:00:00Z,2012-12-05T09:00:00Z,2012-03-08T01:14:02Z,ARG,TRAWLERS,4.640278,9149093,2025-10-13T14:42:19Z,-45.2,-66.199997,701097000,MYRDOMA F,8e28a4205-5e25-dc22-83fa-3ef1ae6ed118,FISHING


In [5]:

df1["date"] = pd.to_datetime(df1["date"]) # Convert the str time to datetime
df1 = df1[["date", "lat", "lon", "hours", "flag", "geartype"]] # Select only the necessary columns

df1 = df1.replace(r"^\s*$", np.nan, regex=True) # Replace empty strings " " with NaN
df1 = df1.dropna(subset=["lat", "lon", "date", "flag", "geartype"]) # Drop rows with NAN in future coordinates xarray columns
geartypes_of_interest = ["TRAWLERS", 'SQUID_JIGGER']
df1 = df1[df1['geartype'].isin(geartypes_of_interest)] #We keep only the interesting gears for SWA intl

In [6]:
available_polygon = gpd.read_file("./data/available_SWA_fishing_area.geojson")

gdf1_points = gpd.GeoDataFrame(
    df1,
    geometry=gpd.points_from_xy(df1["lon"], df1["lat"]),  # lon = X, lat = Y
    crs="EPSG:4326"
)

In [7]:
gdf1_filtered = gpd.sjoin(
    gdf1_points,
    available_polygon,
    predicate="intersects",  
    how="inner"
)

In [8]:
gdf1_filtered

,date,lat,lon,hours,flag,geartype,geometry,index_right
1,2012-02-01,-52.4,-58.099998,4.257778,FLK,TRAWLERS,POINT (-58.1 -52.4),0
11,2012-05-01,-51.2,-62.599998,7.086667,ESP,TRAWLERS,POINT (-62.6 -51.2),0
13,2012-01-01,-46.3,-60.799999,12.975278,KOR,SQUID_JIGGER,POINT (-60.8 -46.3),0
23,2012-06-01,-50.4,-61.299999,9.133056,FLK,TRAWLERS,POINT (-61.3 -50.4),0
27,2012-05-01,-52.6,-63.599998,3.098056,ARG,TRAWLERS,POINT (-63.6 -52.6),0
...,...,...,...,...,...,...,...,...
2853681,2024-05-01,-49.5,-61.099998,7.418889,KOR,SQUID_JIGGER,POINT (-61.1 -49.5),0
2853683,2024-07-01,-46.9,-60.700001,13.288889,CHN,TRAWLERS,POINT (-60.7 -46.9),0
2853689,2024-09-01,-45.8,-60.599998,2.036389,FLK,TRAWLERS,POINT (-60.6 -45.8),0
2853693,2024-06-01,-45.5,-60.599998,12.923889,ESP,TRAWLERS,POINT (-60.6 -45.5),0


In [9]:
df = gdf1_filtered.drop(columns=["geometry", "index_right"]).copy()
df_agg = (
    df.groupby(["date", "lat", "lon", "flag", "geartype"], as_index=False)
      .agg(hours=('hours', 'sum'))
)

df_agg["lat"] = df_agg["lat"].round(1)
df_agg["lon"] = df_agg["lon"].round(1)

In [10]:
dsxr = (
    df_agg
    .set_index(["date", "lat", "lon", "flag", "geartype"])
    .to_xarray()
    .rename({"date": "time"})
)

# Define chunk sizes for dask, time in 12-month chunks, lat and lon full size, geartype and flag one at a chunk
chunks = {"time": 12, "lat": -1, "lon": -1, "geartype":1, "flag":1}
dsxr = dsxr.chunk(chunks)

In [11]:
dsxr

<xarray.Dataset> Size: 2GB
Dimensions:   (time: 156, lat: 236, lon: 151, flag: 27, geartype: 2)
Coordinates:
  * time      (time) datetime64[ns] 1kB 2012-01-01 2012-02-01 ... 2024-12-01
  * lat       (lat) float64 2kB -60.0 -59.9 -59.8 -59.7 ... -36.3 -36.2 -35.9
  * lon       (lon) float64 1kB -65.0 -64.9 -64.8 -64.7 ... -50.2 -50.1 -50.0
  * flag      (flag) object 216B 'ALB' 'ARE' 'ARG' 'BLZ' ... 'UKR' 'URY' 'VUT'
  * geartype  (geartype) object 16B 'SQUID_JIGGER' 'TRAWLERS'
Data variables:
    hours     (time, lat, lon, flag, geartype) float64 2GB dask.array<chunksize=(12, 236, 151, 1, 1), meta=np.ndarray>

In [12]:
lat_min_xr = dsxr.lat.min().item()
lat_max_xr = dsxr.lat.max().item()
lon_min_xr = dsxr.lon.min().item()
lon_max_xr = dsxr.lon.max().item()
bbox_xr = [lon_min_xr, lat_min_xr, lon_max_xr, lat_max_xr] #bb0x from GFW xarray

min_lon, min_lat, max_lon, max_lat = [-69.61, -60., -50., -32.45] #SW Atlantic bbox (roundeed to the second decimal)
bbox_SWA = [min_lon, min_lat, max_lon, max_lat]

print("Bounding box from xarray dataset:", bbox_xr)
print("SW Atlantic bounding box:", bbox_SWA) #Southwest Atlantic bounding box is greater, we need to expand, we will mantain 0.1 degree resolution

lat_diff = np.diff(dsxr.lat)
lon_diff = np.diff(dsxr.lon)

print("Latitude spacing unique values:", np.unique(lat_diff)) #la resolucion es 0.1 grados
print("Longitude spacing unique values:", np.unique(lon_diff))

Bounding box from xarray dataset: [-65.0, -60.0, -50.0, -35.9]
SW Atlantic bounding box: [-69.61, -60.0, -50.0, -32.45]
Latitude spacing unique values: [0.1 0.1 0.2 0.2 0.3]
Longitude spacing unique values: [0.1 0.1 0.1]


In [13]:
#aumentamos el area rellenando con NaN para obtener el area al bbox del estudio

target_lons = np.arange(bbox_SWA[0], bbox_SWA[2] + 0.1, 0.1)
target_lats = np.arange(bbox_SWA[1], bbox_SWA[3] + 0.1, 0.1)

dsxr_expanded = dsxr.reindex(
    lat=target_lats,
    lon=target_lons
)

dsxr_expanded = dsxr_expanded.chunk({
    "time": 12, # 12 months at a chunk
    "lat": -1, # full size
    "lon": -1,
    "flag": 1, # one at a chunk
    "geartype": 1
}) #we need to call chunk again after reindexing (for saving the zarr)

c:\Users\rubar\miniconda3\envs\TFMenv\lib\site-packages\dask\array\core.py:5189: PerformanceWarning: Increasing number of chunks by factor of 26
  result = blockwise(


In [14]:
#Adding metadata and description to this zarr dataset
dsxr_expanded.attrs['title'] = "SW Atlantic GFW AIS Fishing Effort Data (2012-2024)"
dsxr_expanded.attrs["description"] = (
    """Monthly fishing hours per grid cell in the SW Atlantic, 
    aligned to 0.1° lat/lon grid
    only data from trawlers and squid jiggers vessels included,
    only data within the available fishing area is included which comprises
    international waters and claimed Falkland Islands EEZ."""
)

dsxr_expanded.attrs["source"] = "AIS-based fishing effort dataset"
dsxr_expanded.attrs["created_by"] = "Ruben Barriuso"
dsxr_expanded.attrs["resolution"] = "0.1x0.1 degree"
dsxr_expanded.attrs["date_created"] = "2025-12-29"
dsxr_expanded.attrs["source"] = "Global Fishing Watch (GFW)"

dsxr_expanded["hours"].attrs["long_name"] = "Fishing effort in AIS hours"
dsxr_expanded["hours"].attrs["description"] = (
    "Total monthly fishing effort observed in each grid cell from GFW"
)

In [16]:

dsxr_expanded.to_zarr("./resources/GFW/GFW_AIS_Intl-FK_Trwl-Jgr_2012-2024.zarr", 
                      consolidated=True)#creates a metadata file for faster access